# Глава 11. Метрики и матрица ошибок 🎯

**После этого урока ты сможешь:**
- построить и прочитать матрицу ошибок (confusion matrix);
- рассчитать precision, recall и F1;
- объяснить, почему accuracy 99% может означать бесполезную модель;
- выбрать правильную метрику под задачу.

> ⚠️ **Важно про Colab.** Этот ноутбук самодостаточный — запускай ячейки **сверху вниз**.
> Если открыть новый блокнот и запустить сразу код с `full.predict(...)`, будет ошибка
> `NameError: name 'full' is not defined`, потому что модель `full` создаётся в ячейке ниже.
> Кнопка «Среда выполнения → Перезапустить среду» стирает все переменные — тогда запусти всё заново с начала.

## Шаг 0. Провокация: 99% точности — и полная бесполезность

Болезнь встречается у 1 человека из 100. Сделаем «модель», которая ВСЕМ говорит «здоров». Она угадает 99 из 100 — accuracy 99%! Но ни одного больного не найдёт. Отсюда мораль: **accuracy врёт на несбалансированных данных.** Ниже разберёмся, чем её заменить.

## Шаг 1. Готовим данные и обучаем Pipeline `full` (краткая версия главы 10)

Берём «Титаник» прямо из интернета. Собираем тот же конвейер, что в главе 10: заполнение пропусков → масштабирование чисел → кодирование категорий → случайный лес. Аналогия Pipeline — **конвейер на заводе**: данные едут по ленте и на каждой станции обрабатываются.

In [ ]:
import pandas as pd                                        # Pandas — «Excel внутри Python»
from sklearn.model_selection import train_test_split         # делит данные на train/test
from sklearn.pipeline import Pipeline                         # сам конвейер
from sklearn.compose import ColumnTransformer                 # разные обработки для разных столбцов
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer                      # заполнение пропусков
from sklearn.ensemble import RandomForestClassifier          # «голосование класса» — лес деревьев

# 1) Загружаем датасет «Титаник» по ссылке (интернет в Colab есть)
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

# 2) Признаки (X) и цель (y): предсказываем, выжил пассажир или нет
X = df[["Age", "Fare", "Sex", "Pclass"]]   # 3 числовых + 1 текстовый столбец
y = df["Survived"]                          # 1 = выжил, 0 = нет

# 3) Делим на обучающую и тестовую части. random_state=42 — чтобы разбиение было ВОСПРОИЗВОДИМЫМ
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# 4) Описываем, какие столбцы числовые, а какие категориальные
num = ["Age", "Fare", "Pclass"]
cat = ["Sex"]

# 5) Препроцессор: для чисел — заполнить медианой и отмасштабировать;
#    для текста — заполнить самым частым значением и закодировать в 0/1
prep = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc",  StandardScaler())]), num),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh",  OneHotEncoder(handle_unknown="ignore"))]), cat),
])

# 6) Полный конвейер: препроцессор + модель. Обучаем на train.
full = Pipeline([("prep", prep),
                 ("clf",  RandomForestClassifier(random_state=42))]).fit(X_train, y_train)

print("Готово! Pipeline 'full' обучен. Точность на тесте:", round(full.score(X_test, y_test), 3))

## Шаг 2. Матрица ошибок (confusion matrix)

Матрица ошибок раскладывает ответы модели на 4 клетки:

| | Модель сказала «да» | Модель сказала «нет» |
|---|---|---|
| На самом деле «да» | верное попадание (TP) | **пропуск (FN)** |
| На самом деле «нет» | **ложная тревога (FP)** | верный отказ (TN) |

По диагонали — правильные ответы, вне диагонали — два типа ошибок.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# full, X_test, y_test уже существуют — мы создали их в Шаге 1
pred = full.predict(X_test)                 # предсказания модели на тесте
cm = confusion_matrix(y_test, pred)         # считаем 4 клетки матрицы

ConfusionMatrixDisplay(cm, display_labels=["Не выжил", "Выжил"]).plot(cmap="Blues")
plt.title("Матрица ошибок")
plt.show()

print("Матрица ошибок (строки — реальность, столбцы — предсказание):")
print(cm)

## Шаг 3. Precision, Recall и F1

| Метрика | Отвечает на вопрос | Когда важна |
|---|---|---|
| **Precision** | Из тех, кого назвали «да», сколько реально «да»? | когда дорога ложная тревога |
| **Recall** | Из всех реальных «да» сколько нашли? | когда дорого пропустить |
| **F1** | баланс precision и recall | когда важно и то, и другое |

**Аналогия 💡:** спам-фильтр бережёт **precision** (лучше пропустить пару спам-писем, чем отправить важное письмо в спам), а детектор болезни бережёт **recall** (лучше лишний раз перепроверить здорового, чем пропустить больного).

In [ ]:
from sklearn.metrics import classification_report

# Одной командой — все метрики по каждому классу
print(classification_report(y_test, pred, target_names=["Не выжил", "Выжил"]))

## Задания

**Базовый уровень.** Поменяй в Шаге 1 набор признаков (например, добавь `SibSp`, `Parch`) и снова построй матрицу ошибок и `classification_report`. Сколько стало пропусков (FN)?

**Продвинутый.** Сравни accuracy и recall для класса «Выжил». Совпадают ли они? Почему?

**Со звёздочкой ⭐.** Для трёх задач — фильтр спама, детектор болезни, рекомендация фильма — реши, какая метрика (precision / recall / F1) главная, и обоснуй.

## Проверь себя ℹ️
1. Почему accuracy врёт на несбалансированных данных?
2. Чем precision отличается от recall?
3. Для детектора мошеннических операций какая метрика важнее и почему?